# Data Loaders — Tennis Tracker

A **DataLoader** is how PyTorch gets data into your model during training.
Before we can train anything, we need to tell PyTorch:
- Where the data lives on disk
- How to read one sample
- What transformations to apply (resize, normalise, etc.)

PyTorch gives us a base class called `torch.utils.data.Dataset`.
We subclass it and implement three methods:
- `__init__` → read file paths / annotations into memory
- `__len__` → how many samples total
- `__getitem__(i)` → load and return the i-th sample

PyTorch's `DataLoader` wraps our `Dataset` and handles batching, shuffling, and parallel loading automatically.

We have **three datasets** to handle:
1. **Player Detection** (COCO format) → images + bounding boxes → feeds the CNN / detector
2. **Ball Trajectory** (CSV + sequential frames) → sequences of (x,y) positions → feeds the LSTM
3. **Court Keypoints** (COCO keypoint format) → 15 court landmark positions → enables court-relative coordinates & top-down view

In [ ]:
import json
import os
import csv
from pathlib import Path
from collections import defaultdict

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT       = Path("../Data")
PLAYER_DIR = ROOT / "Tennis Player Detection.v5i.coco"
BALL_DIR   = ROOT / "TennisBallDataset"
COURT_DIR  = ROOT / "tennis court.v8-7-court-dataset-missing-keypoint-exc-no-resize-.coco"

print("Player dir exists:", PLAYER_DIR.exists())
print("Ball dir exists  :", BALL_DIR.exists())
print("Court dir exists :", COURT_DIR.exists())

---
## Part 1 — Player Detection Dataset (COCO format)

### What is COCO format?

COCO (Common Objects in Context) is the most popular format for object detection.
Everything lives in one JSON file with this structure:

```
{
  "images":      [ { id, file_name, width, height } ],
  "annotations": [ { id, image_id, category_id, bbox: [x, y, w, h] } ],
  "categories":  [ { id, name } ]
}
```

The key relationship: each annotation has an `image_id` that links it to an image.
One image can have multiple annotations (= multiple players in the frame).

The `bbox` is `[x, y, width, height]` in **pixels**, where (x,y) is the **top-left corner**.

Let's load the JSON and see what we're dealing with.

In [ ]:
ann_path = PLAYER_DIR / "train" / "_annotations.coco.json"

with open(ann_path) as f:
    coco = json.load(f)

print(f"Images      : {len(coco['images'])}")
print(f"Annotations : {len(coco['annotations'])}")
print(f"Categories  : {coco['categories']}")
print()
print("Sample image entry:")
print(coco['images'][0])
print()
print("Sample annotation entry:")
print(coco['annotations'][0])

### Visualise one image with its bounding boxes

Before writing any model code it's always worth plotting a sample.
If the boxes look wrong here, nothing downstream will work.

We'll:
1. Pick an image
2. Find all annotations that belong to it (`annotation['image_id'] == image['id']`)
3. Draw each `bbox` as a rectangle

In [ ]:
ann_by_image = defaultdict(list)
for ann in coco['annotations']:
    ann_by_image[ann['image_id']].append(ann)

sample_img_meta = next(img for img in coco['images'] if ann_by_image[img['id']])

img_path = PLAYER_DIR / "train" / sample_img_meta['file_name']
pil_img  = Image.open(img_path).convert("RGB")
anns     = ann_by_image[sample_img_meta['id']]

fig, ax = plt.subplots(1, figsize=(8, 8))
ax.imshow(pil_img)

for ann in anns:
    x, y, w, h = ann['bbox']
    rect = patches.Rectangle((x, y), w, h,
                              linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y - 4, 'player', color='lime', fontsize=9, backgroundcolor='black')

ax.set_title(f"{sample_img_meta['file_name']}  |  {len(anns)} player(s)")
ax.axis('off')
plt.tight_layout()
plt.show()

### Build the PyTorch Dataset class

Now we formalise what we just did manually into a reusable `Dataset`.

**Design decisions:**
- We normalise bounding boxes from pixels → **relative** (0–1) values.
  Why? Because if we resize images the pixel coordinates change, but relative ones don't.
- We apply a basic image transform: resize to 416×416 and convert to a tensor.
- We skip images with zero annotations so we don't feed empty targets to the model.

In [ ]:
class PlayerDetectionDataset(Dataset):
    """COCO-format dataset for tennis player bounding-box detection."""

    def __init__(self, split: str = "train", img_size: int = 416):
        self.img_dir  = PLAYER_DIR / split
        self.img_size = img_size

        with open(self.img_dir / "_annotations.coco.json") as f:
            coco = json.load(f)

        ann_by_image = defaultdict(list)
        for ann in coco['annotations']:
            ann_by_image[ann['image_id']].append(ann['bbox'])

        self.samples = [
            (img['file_name'], img['width'], img['height'], ann_by_image[img['id']])
            for img in coco['images'] if ann_by_image[img['id']]
        ]

        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_name, orig_w, orig_h, bboxes = self.samples[idx]
        img = Image.open(self.img_dir / file_name).convert("RGB")
        img_tensor = self.transform(img)

        boxes = [[x/orig_w, y/orig_h, w/orig_w, h/orig_h] for x,y,w,h in bboxes]
        target = {
            "boxes":  torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.zeros(len(boxes), dtype=torch.long),
        }
        return img_tensor, target


print("PlayerDetectionDataset defined.")

In [ ]:
train_ds = PlayerDetectionDataset(split="train")
valid_ds = PlayerDetectionDataset(split="valid")

print(f"Train samples : {len(train_ds)}")
print(f"Valid samples : {len(valid_ds)}")

img_t, target = train_ds[0]
print(f"Image tensor shape : {img_t.shape}")
print(f"Boxes              : {target['boxes']}")

In [ ]:
def denorm(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

img_t, target = train_ds[3]
disp = denorm(img_t).permute(1, 2, 0).numpy()
H, W = img_t.shape[1], img_t.shape[2]

fig, ax = plt.subplots(1, figsize=(7, 7))
ax.imshow(disp)
for box in target['boxes']:
    x, y, w, h = box.tolist()
    rect = patches.Rectangle((x*W, y*H), w*W, h*H,
                              linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
ax.set_title("PlayerDetectionDataset — normalised boxes")
ax.axis('off')
plt.tight_layout()
plt.show()

### Wrapping with DataLoader

`DataLoader` takes your `Dataset` and adds batching, shuffling, and parallel loading.

One tricky thing with detection: each image has a *different number of boxes*.
PyTorch's default collation stacks tensors, which requires the same shape.
We use a custom `collate_fn` that returns a list of targets instead of stacking them.

In [ ]:
def detection_collate(batch):
    """Stack images; keep targets as a list (variable boxes per image)."""
    images  = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=0, collate_fn=detection_collate)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False,
                          num_workers=0, collate_fn=detection_collate)

imgs, targets = next(iter(train_loader))
print(f"Batch images shape  : {imgs.shape}")
print(f"Num targets in batch: {len(targets)}")
print(f"First image boxes   : {targets[0]['boxes']}")

---
## Part 2 — Ball Trajectory Dataset (LSTM input)

### What is sequence data?

An LSTM takes a **sequence** of inputs and learns patterns across time.
For us: if we feed it the ball's last N positions, it should predict where the ball goes next.

**The sliding window trick:**
Given a clip of T frames, we create many overlapping sequences of length `seq_len`.
For each sequence, the input is frames `[i … i+seq_len-1]` and the target is frame `[i+seq_len]`.

```
frames:  0  1  2  3  4  5  6  7  8  9
seq_len = 4

sample 0: input=[0,1,2,3]  target=4
sample 1: input=[1,2,3,4]  target=5
...
```

In [ ]:
sample_csv = BALL_DIR / "game1" / "Clip1" / "Label.csv"

rows = []
with open(sample_csv) as f:
    for row in csv.DictReader(f):
        rows.append(row)

print(f"Total frames in clip : {len(rows)}")
print(f"Columns              : {list(rows[0].keys())}")
print()
for r in rows[:5]:
    print(" ", r)

vis_counts = {}
for r in rows:
    v = r['visibility']
    vis_counts[v] = vis_counts.get(v, 0) + 1
print("\nVisibility value counts:", vis_counts)

In [ ]:
xs, ys = [], []
for r in rows:
    if r['visibility'] != '0' and r['x-coordinate']:
        xs.append(float(r['x-coordinate']))
        ys.append(float(r['y-coordinate']))

plt.figure(figsize=(9, 5))
plt.plot(xs, ys, 'o-', markersize=3, linewidth=0.8, color='darkorange')
plt.scatter(xs[0],  ys[0],  color='green', s=80, zorder=5, label='start')
plt.scatter(xs[-1], ys[-1], color='red',   s=80, zorder=5, label='end')
plt.gca().invert_yaxis()
plt.xlabel("x (pixels)")
plt.ylabel("y (pixels)")
plt.title("Ball trajectory — game1/Clip1")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Visible frames: {len(xs)} / {len(rows)}")

### Build the Ball Trajectory Dataset

**Design decisions:**
- Normalise (x,y) to [0,1] relative to frame size — model never sees raw pixels
- Skip frames where ball is not visible (`visibility == 0`)
- `seq_len=8`: feed 8 positions, predict the 9th (tunable)

In [ ]:
sample_frame = BALL_DIR / "game1" / "Clip1" / "0000.jpg"
print("Frame size (W × H):", Image.open(sample_frame).size)

In [ ]:
class BallTrajectoryDataset(Dataset):
    """
    Sliding-window dataset over ball (x,y) trajectories.
    input  : (seq_len, 2) normalised (x,y) positions
    target : (2,)         next position to predict
    """
    IMG_W, IMG_H = 1280, 720

    def __init__(self, ball_dir: Path = BALL_DIR, seq_len: int = 8):
        self.seq_len = seq_len
        self.windows = []

        for game in sorted(ball_dir.iterdir()):
            if not game.is_dir(): continue
            for clip in sorted(game.iterdir()):
                lf = clip / "Label.csv"
                if not lf.exists(): continue
                pos = self._load_clip(lf)
                for i in range(len(pos) - seq_len):
                    self.windows.append((pos[i:i+seq_len], pos[i+seq_len]))

    def _load_clip(self, label_file: Path):
        coords = []
        with open(label_file) as f:
            for row in csv.DictReader(f):
                if row['visibility'] == '0' or not row['x-coordinate']: continue
                coords.append([float(row['x-coordinate']) / self.IMG_W,
                               float(row['y-coordinate']) / self.IMG_H])
        return torch.tensor(coords, dtype=torch.float32)

    def __len__(self):        return len(self.windows)
    def __getitem__(self, i): return self.windows[i]


print("BallTrajectoryDataset defined.")

In [ ]:
ball_ds = BallTrajectoryDataset(seq_len=8)
print(f"Total windows : {len(ball_ds)}")

inp, tgt = ball_ds[0]
print(f"Input shape   : {inp.shape}  (seq_len, 2)")
print(f"Target shape  : {tgt.shape}  (next position)")

ball_loader = DataLoader(ball_ds, batch_size=32, shuffle=True, num_workers=0)
ib, tb = next(iter(ball_loader))
print(f"Batch input   : {ib.shape}")
print(f"Batch target  : {tb.shape}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, idx in zip(axes, [0, 50, 200]):
    inp, tgt = ball_ds[idx]
    ax.plot(inp[:,0].numpy(), inp[:,1].numpy(), 'o-', color='royalblue', label='input')
    ax.scatter(tgt[0].item(), tgt[1].item(), color='red', s=120, zorder=5, label='target')
    ax.set_xlim(0, 1); ax.set_ylim(1, 0)
    ax.set_title(f"Window #{idx}")
    ax.legend(fontsize=7)
plt.suptitle("BallTrajectoryDataset — input (blue) + target (red)")
plt.tight_layout()
plt.show()

---
## Part 3 — Court Keypoint Dataset

### What is keypoint detection?

Instead of predicting a rectangle around an object, keypoint detection predicts the **exact pixel location** of specific named points — like the corners of a tennis court or the joints of a human body.

Our court dataset has **15 keypoints per image**, stored in COCO's keypoint format:
```
keypoints: [x0, y0, v0,  x1, y1, v1,  ...,  x14, y14, v14]
```
That's 45 numbers per annotation — x, y, and visibility (v=2 means visible) for each of the 15 points.

### Why does this matter for us?

Once we can detect those 15 court landmarks, we can compute a **homography** — a mathematical transform that maps the camera's perspective view to a flat, top-down diagram of the court. This lets us:
- Plot players and the ball on a 2D court map in the GUI
- Feed the LSTM **court-relative coordinates** instead of raw pixels (so it generalises across different camera angles)

This is what separates a basic project from a professional-grade one.

### What does our model need to output?

A CNN that takes a 640×640 image and outputs a vector of **30 numbers** — the (x,y) of each of the 15 keypoints, normalised to [0,1]. We use a regression head (MSE loss), not classification.

In [ ]:
# Load and inspect the court annotation JSON
court_ann_path = COURT_DIR / "train" / "_annotations.coco.json"
with open(court_ann_path) as f:
    court_coco = json.load(f)

print(f"Images      : {len(court_coco['images'])}")
print(f"Annotations : {len(court_coco['annotations'])}")
print()

# The category tells us the keypoint names and skeleton connections
cat = court_coco['categories'][1]  # index 1 is 'tennis_court'
print(f"Category    : {cat['name']}")
print(f"Num keypoints: {len(cat['keypoints'])}")
print(f"Keypoint names: {cat['keypoints']}")
print()

# Look at one annotation
ann = court_coco['annotations'][0]
kps = ann['keypoints']  # flat list: [x0, y0, v0, x1, y1, v1, ...]
print(f"Raw keypoints length: {len(kps)}  (= 15 points × 3 values)")
print()

# Reshape to (15, 3) to read it more easily
kps_arr = np.array(kps).reshape(15, 3)
print("Keypoints as (x, y, visibility):")
for i, (x, y, v) in enumerate(kps_arr):
    print(f"  point {i:2d}: x={x:.1f}  y={y:.1f}  vis={int(v)}")

In [ ]:
# Visualise a sample image with its 15 court keypoints overlaid
sample_img_meta = court_coco['images'][0]
img_path = COURT_DIR / "train" / sample_img_meta['file_name']
pil_img  = Image.open(img_path).convert("RGB")

ann      = court_coco['annotations'][0]
kps_arr  = np.array(ann['keypoints']).reshape(15, 3)  # (15, 3)

fig, ax = plt.subplots(1, figsize=(12, 7))
ax.imshow(pil_img)

# Draw the skeleton connections defined in the category
skeleton = cat['skeleton']
for a, b in skeleton:
    # skeleton indices are 1-based in COCO
    xa, ya, va = kps_arr[a - 1]
    xb, yb, vb = kps_arr[b - 1]
    if va > 0 and vb > 0:
        ax.plot([xa, xb], [ya, yb], 'w-', linewidth=1.5, alpha=0.6)

# Draw each keypoint
colours = plt.cm.hsv(np.linspace(0, 0.85, 15))
for i, (x, y, v) in enumerate(kps_arr):
    if v > 0:
        ax.scatter(x, y, color=colours[i], s=80, zorder=5)
        ax.text(x + 8, y, str(i), color=colours[i], fontsize=8, fontweight='bold')

# Draw the bounding box too
bx, by, bw, bh = ann['bbox']
rect = patches.Rectangle((bx, by), bw, bh,
                          linewidth=1.5, edgecolor='cyan', facecolor='none', linestyle='--')
ax.add_patch(rect)

ax.set_title(f"Court keypoints (15 points)  |  {sample_img_meta['file_name']}")
ax.axis('off')
plt.tight_layout()
plt.show()

### Build the Court Keypoint Dataset

**Design decisions:**
- Target is a flat vector of **30 values** — (x,y) for each of 15 keypoints, normalised to [0,1].
  We drop the visibility flag because our model doesn't need to predict it — all points in the training data have `v=2` (fully visible).
- Images are resized to **640×640** — larger than the player dataset because court lines are fine-grained and need resolution to detect accurately.
- We also return the **visibility mask** as a tensor so the loss function can ignore truly occluded points later.

In [ ]:
class CourtKeypointDataset(Dataset):
    """
    COCO-keypoint dataset for tennis court landmark detection.
    Each sample:
      image      : (3, img_size, img_size) normalised tensor
      keypoints  : (15, 2) normalised (x,y) in [0,1]
      visibility : (15,)  float — 1.0 if point is visible, 0.0 if not
    """
    NUM_KP = 15

    def __init__(self, split: str = "train", img_size: int = 640):
        self.img_dir  = COURT_DIR / split
        self.img_size = img_size

        with open(self.img_dir / "_annotations.coco.json") as f:
            coco = json.load(f)

        # Build image_id → annotation lookup (1 court per image, so at most 1 ann)
        ann_by_image = {ann['image_id']: ann for ann in coco['annotations']}

        self.samples = [
            (img['file_name'], img['width'], img['height'], ann_by_image[img['id']])
            for img in coco['images']
            if img['id'] in ann_by_image
        ]

        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_name, orig_w, orig_h, ann = self.samples[idx]

        img = Image.open(self.img_dir / file_name).convert("RGB")
        img_tensor = self.transform(img)

        # Parse keypoints: flat [x0,y0,v0, x1,y1,v1, ...] → (15, 3)
        kps = np.array(ann['keypoints'], dtype=np.float32).reshape(self.NUM_KP, 3)

        # Normalise x,y to [0,1]
        kps[:, 0] /= orig_w  # x
        kps[:, 1] /= orig_h  # y

        keypoints  = torch.tensor(kps[:, :2], dtype=torch.float32)  # (15, 2)
        visibility = torch.tensor(
            (kps[:, 2] > 0).astype(np.float32), dtype=torch.float32  # (15,)
        )

        return img_tensor, keypoints, visibility


print("CourtKeypointDataset defined.")

In [ ]:
court_train = CourtKeypointDataset(split="train")
court_valid = CourtKeypointDataset(split="valid")

print(f"Train samples : {len(court_train)}")
print(f"Valid samples : {len(court_valid)}")

img_t, kps, vis = court_train[0]
print(f"\nImage shape      : {img_t.shape}")
print(f"Keypoints shape  : {kps.shape}  (15 points, x and y each normalised 0-1)")
print(f"Visibility shape : {vis.shape}")
print(f"\nFirst 5 keypoints (normalised x,y):\n{kps[:5]}")
print(f"All visible? {vis.all().item()}  (sum={vis.sum().item()}/15)")

In [ ]:
# Visualise: un-normalise and overlay keypoints on the resized image
img_t, kps, vis = court_train[5]
disp = denorm(img_t).permute(1, 2, 0).numpy()
H, W = img_t.shape[1], img_t.shape[2]

fig, ax = plt.subplots(1, figsize=(10, 7))
ax.imshow(disp)

colours = plt.cm.hsv(np.linspace(0, 0.85, 15))
for i, ((x, y), v) in enumerate(zip(kps.tolist(), vis.tolist())):
    if v > 0:
        ax.scatter(x * W, y * H, color=colours[i], s=100, zorder=5)
        ax.text(x * W + 5, y * H, str(i), color=colours[i], fontsize=9, fontweight='bold')

ax.set_title("CourtKeypointDataset — 15 landmarks on resized 640×640 image")
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# DataLoader — shapes are fixed (always 15 keypoints) so default collation works
court_loader = DataLoader(court_train, batch_size=8, shuffle=True, num_workers=0)

imgs, kps_batch, vis_batch = next(iter(court_loader))
print(f"Image batch shape      : {imgs.shape}       (B, C, H, W)")
print(f"Keypoints batch shape  : {kps_batch.shape}   (B, 15, 2)")
print(f"Visibility batch shape : {vis_batch.shape}    (B, 15)")
print()
print("When training the keypoint model, the loss will be:")
print("  MSE between predicted (B,15,2) and target (B,15,2),")
print("  multiplied element-wise by visibility so occluded points don't contribute.")

---
## Summary

You now have three working PyTorch datasets:

| Dataset | `__getitem__` returns | Shape | Used by |
|---|---|---|---|
| `PlayerDetectionDataset` | image + bbox dict | `(3,416,416)` + `(N,4)` | CNN / Detector |
| `BallTrajectoryDataset` | sequence + next pos | `(8,2)` + `(2,)` | LSTM |
| `CourtKeypointDataset` | image + keypoints + visibility | `(3,640,640)` + `(15,2)` + `(15,)` | Keypoint CNN |

**Next step:** copy the three classes into `utils/datasets.py` so any other module can do `from utils.datasets import PlayerDetectionDataset` — then start building the CNN in `modules/cnn/`.